# Snowflake Python APIs 101

[Snowflake Python APIs: Managing Snowflake objects with Python | Snowflake Documentation](https://docs.snowflake.com/en/developer-guide/snowflake-python-api/snowflake-python-overview)

[Common setup for Snowflake Python APIs tutorials | Snowflake Documentation](https://docs.snowflake.com/en/developer-guide/snowflake-python-api/tutorials/common-setup#set-up-your-development-environment)

[Snowflake Python APIs: Managing Snowflake objects with Python | Snowflake Documentation](https://docs.snowflake.com/en/developer-guide/snowflake-python-api/snowflake-python-overview)


## Install Snowflake Python Package 



## Setup development environment 

https://docs.snowflake.com/en/developer-guide/snowflake-python-api/tutorials/common-setup#set-up-your-development-environment

Create a file named `$HOME/.snowflake/connections.toml` with the following connection parameters, and update it with your real credentials:

```toml
[default]
account = "<YOUR ACCOUNT NAME>"
user = "<YOUR ACCOUNT USER>"
password = "<YOUR ACCOUNT USER PASSWORD>"
# optional
# warehouse = "<YOUR COMPUTE WH>"
# optional
# database = "<YOUR DATABASE>"
# optional
# schema = "<YOUR SCHEMA>"
```

## Load packages

In [ ]:
from datetime import timedelta

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col
from snowflake.core import Root, CreateMode
from snowflake.core.database import Database
from snowflake.core.schema import Schema
from snowflake.core.stage import Stage
from snowflake.core.table import Table, TableColumn, PrimaryKey
from snowflake.core.task import StoredProcedureCall, Task
from snowflake.core.task.dagv1 import DAGOperation, DAG, DAGTask
from snowflake.core.warehouse import Warehouse

from snowflake.core.user import User
from snowflake.core.role import Role
from snowflake.core.user import Securable


from rich import print as rprint

## Set constants    

In [ ]:
CONNECTION_NAME= "<YOUR-VALUE-HERE>"
DATABASE_NAME="PYTHON_API_DB"
DATABASE_SCHEMA="PYTHON_API_SCHEMA"
TABLE_NAME="PYTHON_API_TABLE"

WAREHOUSE_NAME="PYTHON_API_WH"
WAREHOUSE_SIZE="SMALL"


In [ ]:
session = Session.builder.config("connection_name", CONNECTION_NAME).create()

In [ ]:
root = Root(session)

## Create database

In [ ]:
database = root.databases.create(
  Database(
    name=DATABASE_NAME),
    mode=CreateMode.or_replace
  )

## Create scheme in database


In [ ]:
schema = database.schemas.create(
  Schema(
    name=DATABASE_SCHEMA),
    mode=CreateMode.or_replace,
  )

## Create table in database 

In [ ]:
table = schema.tables.create(
  Table(
    name=TABLE_NAME,
    columns=[
      TableColumn(
        name="TEMPERATURE",
        datatype="int",
        nullable=False,
      ),
      TableColumn(
        name="LOCATION",
        datatype="string",
      ),
    ],
  ),
mode=CreateMode.or_replace
)

## Check object data

In [ ]:
table_details = table.fetch()

In [ ]:
table_data = table_details.to_dict()
rprint(table_data)

## Alter a table

Append a column called `elevation` to the table.

First create column definition:

In [ ]:
table_details.columns.append(
    TableColumn(
      name="elevation",
      datatype="int",
      nullable=False,
      constraints=[PrimaryKey()],
    )
)

> **Note**  
> This code does not create the column. Instead, this column definition is appended to the array that represents the table’s columns in the `TableModel`. To view this array, review the value of 'columns' as described in the instructions for viewing the table metadata.

In [ ]:
table.create_or_alter(table_details)

In [ ]:
table_data2 = table.fetch().to_dict()
rprint(table_data2)

## Create and manage warehouses

Retrieve collection of warehouse associated in your session

In [ ]:
warehouses = root.warehouses

Creates a warehouse, size and auto-suspend timeout in seconds (500 - 8.3 minutes)


In [ ]:
python_api_wh = Warehouse(
    name=WAREHOUSE_NAME,
    warehouse_size=WAREHOUSE_SIZE,
    auto_suspend=500,
)

warehouse = warehouses.create(python_api_wh, 
                              mode=CreateMode.or_replace)

### Retrieve Warehouse Info

In [ ]:
warehouse_details = warehouse.fetch()
warehouse_data = warehouse_details.to_dict()
rprint(warehouse_data)

## Create User

In [ ]:
my_user = User(name="test_user1")

root.users.create(my_user)

### list users 

https://docs.snowflake.com/en/developer-guide/snowflake-python-api/snowflake-python-managing-user-roles#listing-users

In [ ]:
# users = root.users.iter(like="test_%")
users = root.users.iter()
for user in users:
  print(user.name)

## create role 

https://docs.snowflake.com/en/user-guide/security-access-control-configure#create-a-role 
https://docs.snowflake.com/en/user-guide/security-access-control-configure#grant-privileges-to-a-role 



In [ ]:
my_role = Role(name="test_role1")
root.roles.create(my_role)

### list roles


In [ ]:
role_list = root.roles.iter()
for role_obj in role_list:
  print(role_obj.name)

## grant role to user 

In [ ]:
root.users['test_user1'].grant_role(role_type="ROLE", role=Securable(name='test_role1'))

### get user info 

In [ ]:
my_user = root.users["test_user1"].fetch()
rprint(my_user.to_dict())

## drop user



In [ ]:
my_user_res = root.users["test_user1"]
my_user_res.drop()

### drop role


In [ ]:
my_role_res = root.roles["test_role1"]
my_role_res.drop()

### relist to confirm deletion

In [ ]:
role_list = root.roles.iter()
for role_obj in role_list:
  print(role_obj.name)

In [ ]:
# users = root.users.iter(like="test_%")
users = root.users.iter()
for user in users:
  print(user.name)